# Week 1 – Python for Machine Learning & Data Preprocessing
## HR Employee Dataset

**Author:** Himanshu Raj  
**Dataset:** `HR_Raw_Dataset.csv` (simulated HR data already in this repository)

### Objective
Inspect, clean, transform and explore HR data using Python, NumPy, Pandas and Scikit-learn. This notebook extends the original HR data-cleaning work with feature selection, categorical encoding, numerical standardization, EDA and reproducible train/test preprocessing.

> Educational exercise only. The dataset is synthetic; do not use this workflow for real hiring, promotion, termination or other consequential employment decisions.

## 1. Import libraries


In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')
print('Libraries imported successfully')

## 2. Load and inspect the raw dataset


In [ ]:
possible_paths = [Path('HR_Raw_Dataset.csv'), Path('../HR_Raw_Dataset.csv')]
data_path = next((p for p in possible_paths if p.exists()), None)
if data_path is None:
    raise FileNotFoundError('Place HR_Raw_Dataset.csv in the repository root.')
df_raw = pd.read_csv(data_path)
print('Shape:', df_raw.shape)
display(df_raw.head())
display(df_raw.dtypes.to_frame('Data type'))
display(df_raw.describe(include='all').T)
display(df_raw.isna().sum().to_frame('Missing values'))
print('Duplicate rows:', df_raw.duplicated().sum())

## 3. Clean and validate data
Remove exact duplicates, standardize text labels, convert data types, and impute missing values. Median and mode are used as simple baseline strategies.


In [ ]:
df = df_raw.drop_duplicates().copy()
categorical_cols = ['Gender', 'Department', 'Job_Role', 'Location', 'Employment_Status']
numeric_cols = ['Age', 'Annual_Salary_INR', 'Years_of_Experience', 'Performance_Rating', 'Attendance_Percentage', 'Leave_Days']
for col in categorical_cols:
    df[col] = df[col].astype('string').str.strip()
for col in ['Department', 'Job_Role', 'Location']:
    df[col] = df[col].str.title()
df['Gender'] = df['Gender'].replace({'Non-Binary':'Non-binary', 'Prefer Not To Say':'Prefer not to say'})
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
df['Date_of_Joining'] = pd.to_datetime(df['Date_of_Joining'], errors='coerce')
for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())
for col in categorical_cols:
    df[col] = df[col].fillna('Unknown')
print('Shape after duplicate removal:', df.shape)
display(df.isna().sum().to_frame('Missing values after cleaning'))

## 4. Feature selection and target separation
Employee ID and employee name are direct identifiers and are not used as model features. Employment status is selected only as an example target to demonstrate a supervised preprocessing pipeline.


In [ ]:
selected_features = ['Age', 'Gender', 'Department', 'Job_Role', 'Location', 'Annual_Salary_INR', 'Years_of_Experience', 'Performance_Rating', 'Attendance_Percentage', 'Leave_Days']
X = df[selected_features].copy()
y = df['Employment_Status'].astype('string').copy()
print('Selected features:', selected_features)
display(y.value_counts().to_frame('Records'))

## 5. Train/test split and preprocessing
Split before fitting the imputers, encoder and scaler to avoid data leakage. One-Hot Encoding transforms categories into numeric indicator columns; StandardScaler standardizes numeric features using training-set statistics.


In [ ]:
try:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
except ValueError:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
num_features = X.select_dtypes(include='number').columns.tolist()
cat_features = X.select_dtypes(exclude='number').columns.tolist()
num_pipe = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
cat_pipe = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('encoder', OneHotEncoder(handle_unknown='ignore'))])
preprocessor = ColumnTransformer([('numeric', num_pipe, num_features), ('categorical', cat_pipe, cat_features)])
train_array = preprocessor.fit_transform(X_train)
test_array = preprocessor.transform(X_test)
feature_names = preprocessor.get_feature_names_out()
if hasattr(train_array, 'toarray'):
    train_array = train_array.toarray()
    test_array = test_array.toarray()
X_train_ready = pd.DataFrame(train_array, columns=feature_names, index=X_train.index)
X_test_ready = pd.DataFrame(test_array, columns=feature_names, index=X_test.index)
display(X_train_ready.head())
print('Train shape:', X_train_ready.shape, '| Test shape:', X_test_ready.shape)

## 6. Exploratory Data Analysis (EDA)
The following charts are descriptive summaries of the synthetic dataset; they do not imply causation.


In [ ]:
plt.figure(figsize=(8,4))
sns.histplot(df['Age'], bins=15, kde=True)
plt.title('Employee Age Distribution')
plt.tight_layout()
plt.show()
plt.figure(figsize=(8,4))
sns.countplot(data=df, y='Department', order=df['Department'].value_counts().index)
plt.title('Employee Count by Department')
plt.tight_layout()
plt.show()
plt.figure(figsize=(7,4))
sns.countplot(data=df, x='Employment_Status')
plt.title('Employment Status Distribution')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()
plt.figure(figsize=(9,6))
sns.heatmap(df[numeric_cols].corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.show()

## 7. Export datasets and validate outputs


In [ ]:
repo_root = Path.cwd()
if not (repo_root / 'HR_Raw_Dataset.csv').exists() and (repo_root.parent / 'HR_Raw_Dataset.csv').exists():
    repo_root = repo_root.parent
output_dir = repo_root / 'outputs'
output_dir.mkdir(exist_ok=True)
train_export = X_train_ready.copy()
train_export['Employment_Status'] = y_train.loc[train_export.index].values
test_export = X_test_ready.copy()
test_export['Employment_Status'] = y_test.loc[test_export.index].values
df.to_csv(output_dir / 'HR_Cleaned_Dataset_Week1.csv', index=False)
train_export.to_csv(output_dir / 'HR_Train_Preprocessed.csv', index=False)
test_export.to_csv(output_dir / 'HR_Test_Preprocessed.csv', index=False)
assert X_train_ready.isna().sum().sum() == 0
assert X_test_ready.isna().sum().sum() == 0
print('Export complete:', output_dir.resolve())
print('Raw records:', len(df_raw), '| Cleaned records:', len(df))
print('Train records:', len(X_train_ready), '| Test records:', len(X_test_ready))
print('Processed features:', len(feature_names))

## 8. Learning outcomes and conclusion
This notebook demonstrates data loading, inspection, duplicate removal, missing-value imputation, type conversion, feature selection, One-Hot Encoding, numerical standardization, EDA, train/test splitting, and export. The preprocessing transformations are fitted only on the training set and reused for the test set.

**Responsible-use note:** Employment status is an illustrative target only. Do not use this synthetic exercise to make real employment decisions.